# Phase 6 — Siamese / Dual-Stream U-Net Model Training

This notebook prepares the environment, mounts Google Drive to retrieve model-ready chips, loads the dataset index, and defines a PyTorch Dataset for Phase 6 training.

In [ ]:
# 1. Clone GitHub repository
!git clone https://github.com/Yasir-29/SAR.git
%cd SAR/xbd-s12

In [ ]:
# 2. Install requirements
!pip install -r requirements.txt

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Define dataset paths
import os
from pathlib import Path

# Define where the model_ready directory is located on Google Drive
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/SAR_model_ready')
LOCAL_INDEX_PATH = Path('data/model_ready/dataset_index.csv')
LOCAL_NORM_PATH = Path('data/model_ready/normalization.json')

In [ ]:
# 5. Load dataset_index.csv
import pandas as pd
df_index = pd.read_csv(LOCAL_INDEX_PATH)
df_index.head()

In [ ]:
# 6. Load normalization.json
import json
with open(LOCAL_NORM_PATH, 'r') as f:
    norm_params = json.load(f)
print(json.dumps(norm_params, indent=2))

In [ ]:
# 7. Verify train/validation/test samples & Print class distribution
supervised_df = df_index[df_index['split'] != 'exclude']
print("Total supervised samples:", len(supervised_df))
print("\nSplit counts:\n", supervised_df['split'].value_counts())
print("\nClass counts:\n", supervised_df['damage_label'].value_counts())
print("\nClass distribution percentages:\n", supervised_df['damage_label'].value_counts(normalize=True) * 100)

In [ ]:
# 8. Prepare the PyTorch dataset loader
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class BuildingDamageDataset(Dataset):
    def __init__(self, df_index, split, drive_data_dir, norm_params=None):
        self.samples = df_index[df_index['split'] == split].reset_index(drop=True)
        self.drive_data_dir = Path(drive_data_dir)
        self.norm_params = norm_params
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        row = self.samples.iloc[idx]
        
        # The .npy files should be downloaded/copied from Google Drive to local paths,
        # or loaded directly from the mounted Google Drive path.
        # In this dataset loader we assume they are under drive_data_dir.
        bld_id = row['building_id']
        location = row['location']
        split_name = row['split']
        
        # Construct correct path under Drive
        # Drive structure: DRIVE_DATA_DIR / region / split / {bld_id}_*.npy
        region_dir = self.drive_data_dir / location / split_name
        
        sar_pre = np.load(region_dir / f"{bld_id}_sar_pre.npy")
        sar_post = np.load(region_dir / f"{bld_id}_sar_post.npy")
        opt_pre = np.load(region_dir / f"{bld_id}_optical_pre.npy")
        opt_post = np.load(region_dir / f"{bld_id}_optical_post.npy")
        mask = np.load(region_dir / f"{bld_id}_mask.npy")
        
        # Perform normalization using norm_params if provided
        if self.norm_params:
            # SAR VV (index 0), VH (index 1)
            sar_pre_mean = np.array(self.norm_params['sar_pre_mean'])[:, None, None]
            sar_pre_std = np.array(self.norm_params['sar_pre_std'])[:, None, None]
            sar_pre = (sar_pre - sar_pre_mean) / sar_pre_std
            
            sar_post_mean = np.array(self.norm_params['sar_post_mean'])[:, None, None]
            sar_post_std = np.array(self.norm_params['sar_post_std'])[:, None, None]
            sar_post = (sar_post - sar_post_mean) / sar_post_std
            
            # Optical B2, B3, B4, B8
            opt_pre_mean = np.array(self.norm_params['optical_pre_mean'])[:, None, None]
            opt_pre_std = np.array(self.norm_params['optical_pre_std'])[:, None, None]
            opt_pre = (opt_pre - opt_pre_mean) / opt_pre_std
            
            opt_post_mean = np.array(self.norm_params['optical_post_mean'])[:, None, None]
            opt_post_std = np.array(self.norm_params['optical_post_std'])[:, None, None]
            opt_post = (opt_post - opt_post_mean) / opt_post_std
            
        # Convert to float32 Tensors
        x_sar = torch.from_numpy(np.concatenate([sar_pre, sar_post], axis=0)).float()
        x_opt = torch.from_numpy(np.concatenate([opt_pre, opt_post], axis=0)).float()
        y_mask = torch.from_numpy(mask).float().unsqueeze(0)
        y_label = torch.tensor(int(row['damage_label']), dtype=torch.long)
        
        return {
            'sar': x_sar,       # shape (4, 32, 32)
            'optical': x_opt,   # shape (8, 32, 32)
            'mask': y_mask,     # shape (1, 32, 32)
            'label': y_label    # scalar class: 0, 1, 2
        }

# Example instantiation
# train_dataset = BuildingDamageDataset(df_index, 'train', DRIVE_DATA_DIR, norm_params)
# train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
print("PyTorch BuildingDamageDataset ready!")